# Local sourcing share and amplification

A trimmed copy of `tests_counterfactuals.ipynb` (the full notebook is archived in
`old_version/tests_counterfactuals.ipynb`). Two tests only:

1. **Amplification**: $D_r$, the domestic sales generated by one euro of final demand in $r$.
2. **Local sourcing share**: $L_r(d)$, the share of that upstream response landing within $d$
   of the buyer, as a level and as a dispersion, with the estimated geography decomposed into
   its three forces.

Every economy is simulated in Python from `theta+`, by one forward map.
Each section runs on its own after the imports and the Constants cell.


In [1]:
%pip install matplotlib pandas statsmodels seaborn geopandas pyfixest pyarrow scipy



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# The libraries. Every function this notebook calls lives in one of them, so a section
# is edited in a file rather than in a 1,400-line cell, and the gates under `test/`
# import the same code instead of slicing it out of the notebook's JSON.
#
#   utils.py          the loader, `theta+`, the Ricardian geometry, and THE ECONOMY
#   diffusion_lib.py  amplification, the local share, incidence, the IO benchmark
#   granular_lib.py   comparative advantage, concentration, portfolios, the local share
#                     dispersion
#
# Reload after editing one of them: `import importlib; importlib.reload(utils)` --
# or restart the kernel, which is safer once several modules have changed.
import os
import re
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import utils
import diffusion_lib
import granular_lib
from utils import *
from diffusion_lib import *
from granular_lib import *


In [3]:
# Set the width of your LaTeX document in points

plt.rc('text', usetex=True)
plt.rc('font', family='serif')
cmap = plt.get_cmap('viridis')

toulouse_color = (132/255, 46/255, 27/255)
document_width_pt = 511.
font_size = 20  # Adjust according to your preference
plt.rcParams.update({
    "font.size": font_size,
    "axes.labelsize": font_size,
    "axes.titlesize": font_size+2,
    "xtick.labelsize": font_size-3,
    "ytick.labelsize": font_size-3,
    "legend.fontsize": font_size-3,
    "legend.title_fontsize": font_size,
    "figure.titlesize": font_size,
})

plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}\usepackage{amssymb}",
})


## Constants

In [ ]:
# mu = 1 -> step1 best parameters (theta_hat_1)
# mu = 2 -> step3 best parameters (theta_hat_2, the efficient estimate)
MU = 2

# Joint reporting: both industries in one table, one figure set each.
INDUSTRIES = [{"industry": "auto", "display_name": "Motor Vehicles"},
              {"industry": "aero", "display_name": "Aerospace"}]

# The run tree, and WHICH PARTS of it to read. `parts` is the specialisation: this
# notebook reports no moment fit, no inference and no Jacobian, so it reads none of
# those artefacts and opens a tree that carries none of them. It does not read
# `suppliers.parquet` either -- every economy here is simulated from `theta+`.
#
# Add "firm" to compare against Julia's own realisation with `check_against_julia`.
RUN_KWARGS = dict(base="..", profile_T=True, ca_level="aa",
                  granular=True, relax_n_lo=Fase, optimizer="pso",
                  parts=("core", "geography"))

# How many economies each regime draws. `None` takes `utils.ECONOMY_REPLICATIONS`.
# The Julia-side knob is `POST_HOC_REPLICATIONS` in `main.jl` and is a DIFFERENT
# quantity: it governs how many economies are solved and written for the verification
# artefact, which nothing below reads.
ECONOMY_REPLICATIONS = 1000

# Radii for the amplification section; the FIRST is the headline one.
AMPLIFICATION_RADII = (100, 200)

# The constants above override the module defaults, so push them where the library
# functions will resolve them. (A module reads its OWN globals, which is what makes a
# stub work in the gates; the same mechanism applies here.)
for _m in (utils, diffusion_lib, granular_lib):
    for _k in ("AMPLIFICATION_RADII", "ECONOMY_REPLICATIONS"):
        if hasattr(_m, _k):
            setattr(_m, _k, globals()[_k])

out_folder = "../reporting_combined"
os.makedirs(out_folder, exist_ok=True)


## Loading

`utils.load_granular_data(industry, mu, parts=...)` rebuilds, on the Python side, the
layout `load_parameters.jl` builds under `--granular=true --ca_level=aa`, and reads only
the parts it is asked for. This notebook asks for `("core", "geography")`: it reports no
moment fit, no inference and no Jacobian, so it reads none of those artefacts and opens a
run tree that carries none of them.

`utils.reporting_data(industry, mu, ...)` is the entry point the sections use. It loads
the run, reads `theta+` off it, and simulates every regime in `CF_REGIMES` from that one
parameter set, returning `{regime: data}` where each `data` carries that regime's own
economy in `suppliers.parquet`'s schema. The result is cached, so the sections share one
solve.


# Amplification and local sourcing

Section 5.1 of the paper. A demand shock lands on a downstream region; the model says how
much upstream activity it generates (test 1) and how much of it the region keeps (test 2).

**Conventions that set the LEVEL of $D_r$.** `share` is `exp_val`, a share of the downstream
firm's UNIT COST, without the markup $\mu$. The model has one upstream tier, so $D_r$ is the
whole of its amplification, with no round-2 Leontief term.


## Test 1 — how large is the upstream response?

$D_r$ is the aggregate increase in DOMESTIC sales generated by one additional euro of final
demand in downstream region $r$: the euro itself, plus what it pulls from French upstream
cells. Summing `share` over every supplying French firm gives $r$'s domestic
intermediate-input share, so

$$D_r = 1 + \sum_l X_{lr}.$$

**Part of the extra sales leaks abroad.** The euro raises $r$'s intermediate bill by
$(1-\chi_L)(P_r/c_r)^{1-\lambda}$, but each upstream sector $s$ has a foreign competitor
(Fréchet scale $\Phi^F_s$, common to every buyer) that wins a share
$f_{rs} = \Phi^F_s/\Phi_{rs}$ of the varieties, hence of the spending. Those euros buy
imports and create no French sales. With $\theta_{rs}$ the sector mix of the bill and
$m_r = \sum_s \theta_{rs} f_{rs}$ the import share of the bill, the domestic amplification is

$$D_r = 1 + (1-\chi_L)(P_r/c_r)^{1-\lambda}\,(1 - m_r).$$

Upstream cells use labour only, so there is no further round. The parquet carries French
linkages only, so `1 + sum(share)` IS this domestic coefficient.

**It has a closed form, and that is the test's real content.** Summing `exp_val` over the
varieties of a sector kills the within-sector CES index exactly, and summing over sectors
kills the across-sector one, leaving $D_r = 1 + (1-\chi_L)(P_r/c_r)^{1-\lambda}(1-m_r)$ (and
$1 + (1-\chi_L)(P_r/c_r)^{1-\lambda}$ without imports). So $D_r$
is a COST-SHARE object: the comparative-advantage parameters enter it only through
$P_r$ and the import leakage $m_r$. Two
consequences follow. The identification of $T$ only up to $w^\theta T$ does **not**
contaminate $D_r$ — that conflation shapes the geography of a shock, not its size. And the
only regional term is $(P_r/c_r)^{1-\lambda}$, local access to cheap inputs, which is why the
coefficients come out nearly flat across commuting zones.

The variety-count check runs first. A wrong variety count is not a detail of the input; it is
a different model, and everything below is built on that parquet.


In [5]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    # the ESTIMATED economy, simulated from theta+ -- `reporting_data` caches,
    # so the sections share one solve rather than re-solving each
    data = reporting_data(industry, mu=MU, **RUN_KWARGS)["Both forces"]
    globals().update(data)


    diff = build_diffusion_frame(data)
    summ = amplification_summary(data, radii=AMPLIFICATION_RADII, diffusion=diff)
    print(f"\n  mean D_r = {summ['amplification'].mean():.3f} "
          f"(one euro of demand -> {summ['upstream_sales'].mean():.3f} upstream), "
          f"p90 = {summ['amplification'].quantile(0.9):.3f}")
    display(summ.round(3))

    plot_amplification(data, summary=summ,
                       save_to=f"{out_folder}/amplification_{industry}_mu{MU}.pdf")
    plt.show()


FileNotFoundError: run folder ../reporting_auto_profiled_aa_gran_kappa_nlo1_pso not found — check profile_T / ca_level / granular / relax_n_lo / optimizer against main.jl's naming

## Test 2 — how much of the shock does the region keep, and how reliably?

How far the euro travels is not the same question as how much of it the shocked region
retains, and it is the second that the reduced-form part of the paper measures:

$$L_r(d) = \frac{\sum_{r' : d_{r'r} \le d} X_{r'r}}{\sum_{r'} X_{r'r}}.$$

The denominator is UPSTREAM SALES ONLY — the euro of downstream demand is not in it — so it is
a share of what propagates. The radius is a free parameter throughout; the headline is 200 km,
the radius of the reduced-form evidence, and `local_share_profile` sweeps it so that number is
visibly one point on a CDF rather than assumed robust.

The test has two halves and they are the same statistic read twice: the **level**, region by
region and with each force switched off, and the **dispersion** around it, which exists because
with $N_s$ finite the realised network is a random object. They were written as two tests and
are merged here, since a level without its band and a band without its level are each half an
answer.

### The level, and the sign reversal

**This is the section's sharpest result.** In motor vehicles the two forces reinforce: removing
either lowers the local share. In aerospace they work against each other — equalising $\hat T$
*raises* it. Comparative advantage there is not a localising force at all: it takes the response
off the region where the shock starts and sends it to the hubs. That is precisely the
reallocation the average distance of test 4 nearly cancels, which is why $d_r$ and $L_r$ are
both needed rather than one being a check on the other.

The map is drawn because $L_r(d)$ is a number attached to a PLACE, and a bar chart sorted by
value destroys exactly the geography the result claims. Its colour scale is PINNED and shared
across the pair: the content is that aerospace reaches 0.56 where motor vehicles never passes
0.44, and a per-panel scale would rescale that away and paint the two maps alike.

### The dispersion, and why it separates cleanly

Because $L_r(d)$ is **linear** in the realised network, where the Herfindahl of test 5 bis is
quadratic, its two halves separate instead of having to be decomposed. Writing
$\omega_{lrs}(\Omega)$ for the share of buyer $r$'s sector-$s$ purchases landing on upstream
cell $l$, and taking any indicator $h_l$ over cells,

$$\mathbb{E}_\Omega\Big[\sum_l h_l\,\omega_{lrs}\Big] = p_{rs} := \sum_l h_l\,\gamma_{lrs},
\qquad
\operatorname{Var}_\Omega\Big(\sum_l h_l\,\omega_{lrs}\Big) = V_{rs}\;p_{rs}\,(1-p_{rs}).$$

Both follow from the one kernel the section already uses,
$\mathbb{E}[\omega_l\omega_m] = V\gamma_l\mathbf{1}\{l=m\} + (1-V)\gamma_l\gamma_m$ — the same
object that gives $\mathbb{E}[H] = V + (1-V)H(\gamma)$ — since it makes
$\operatorname{Var}(\sum_l h_l\omega_l) = V\operatorname{Var}_\gamma(h)$, and the variance of an
indicator under $\gamma$ is $p(1-p)$. With $h_l = \mathbf{1}\{l = r\}$ this is the own-zone
share; with $h_l = \mathbf{1}\{d_{lr} \le d\}$ it is the $L_r(d)$ of the half above. The radius
is therefore a reporting choice, not a second derivation — and the one reported is **200 km**,
because the own-zone share is an order of magnitude lower, which flattens the band and leaves
the figure with nothing to read.

**So there is nothing structural-against-granular to split in the level.** The point *is* the
continuum benchmark, exactly, and the whole of granularity sits in the spread. That is why this
is a level opposed to a dispersion and not the stacked decomposition of test 5 bis. It also
buys a free specification check: $\mathbb{E}[\omega] = \gamma$ is exact, so the gap between the
realised mean and the closed-form level is pure simulation error, and a large $z$ means the
variety panel and the structural network are not the same economy.

Aggregated over a buyer's sectors with its input mix $\theta_{rs} = X_{rs}/X_r$, and using that
sectors draw their varieties independently,

$$\bar\omega_{r} = \sum_s \theta_{rs}\,\omega_{rs}, \qquad
\mathbb{E}[\bar\omega_r] = \sum_s \theta_{rs}\,p_{rs}, \qquad
\sigma_r^2 = \sum_s \theta_{rs}^{2}\,V_{rs}\,p_{rs}(1-p_{rs}).$$

The weights enter **squared**, so aggregating ten sectors diversifies most of the chance away
before it reaches the figure. That is a property of the statistic and not of the economy, which
is why the per-sector view is kept and the two sectors carrying the most spending are drawn on
their own.

**The band is a quantile range, not $\pm\sigma$, and that is a statement about the law rather
than about taste.** Where $p_r \times N^{\text{eff}}_r$ is of order one the realised local
share is a count of a handful of events: a buyer at 5% with five effective varieties realises
0 or 0.2, never 0.05. The standard deviation exists but describes that distribution badly — it
is discrete and skewed, and a symmetric bar draws a shape the data do not have. The figure
therefore carries the empirical **10–90 percentiles of the draws**, which impose nothing and
let the asymmetry show.

**What the band is, and what it is not.** Every replication is drawn at the *same* $\theta^+$,
so the band is **granularity**: at these fundamentals, the range within which eight realised
economies out of ten place the statistic. It carries none of the uncertainty in
$(\hat\alpha, \hat T, \hat N)$ — it is not a confidence interval and is never captioned as
one. Two numbers say whether it reads as written. `coverage` is the share of realisations that
actually fall inside $[q_{10}, q_{90}]$: on a continuous law it is 0.80, but the realised share
lives on a **lattice** $\{0, 1/N^{\text{eff}}, 2/N^{\text{eff}}, \dots\}$, so with few atoms
the interval over-covers — up to 1.0 — while its width jumps by whole lattice steps and can
collapse to zero when one atom carries more than 80% of the mass. `p_n_eff` says how fine that
lattice is, and is read first.

**Two readings, and the second is a question the numbers have to settle.** First, $p(1-p)$ is
maximal at $p = 1/2$ and vanishes at both ends, so the buyers whose local sourcing is
*intermediate* are the ones whose realised local share says least: one observed at 60% local in
a given year can sit at 35% the next with no fundamental having moved. Second, cutting trade
costs lowers the point, and the hope is that it *raises* the bar by pushing $\gamma$ towards the
middle — the compensation the Herfindahl shows, chance filling the dispersion geography vacates.
That requires $p$ to be pushed **towards** one half; below it the bar rises and falls *with* the
point, so which statement holds is empirical. `local_share_report` counts the buyers moving each
way and reports the share sitting past $1/2$, so the conjecture is decided rather than
restated.

**Two notes the band must carry.** $\theta_{rs}$ is held at its mean, so the reported
dispersion is the **network's alone**; in the realised economy the sector mix moves with the
draw too, and the observable variability is larger. And these bands are **per buyer**: the industry's aggregate local share needs $\operatorname{Cov}_\Omega$ *between*
buyers — cells (iii) and (iv) of test 5 bis — so its band is not the average of these.

**The decomposition and the layout.** Comparative advantage has two layers,
$T_{ls} = T_{a(l)s}\,n_{ls}^{\kappa}$, so the estimated geography rests on three forces, each
removed alone (`CF_REGIMES_KAPPA`):

| Regime | What changes | What is kept |
|---|---|---|
| Equal area advantage | $T_{a s} = 1$ for every attraction area | density gradient $n_{ls}^{\kappa}$, distance |
| No density gradient | $\kappa = 0$ | area advantage $T_{as}$, distance |
| No trade cost | $\alpha = 0$ | both layers of $T$ |

(The "Equal comparative advantage" regime of the other sections removes both layers at once;
it equals "Equal area advantage" at $\kappa = 0$.) Each buyer's import share is held at its
estimate in every regime, so the comparison is about the domestic network only and does not
depend on the reference area that sets the units of $T$.

Two figures (`layout="pair"`), sharing the buyer order (the estimated local share) and the
$x$ window: **(1)** the estimated economy alone, in black, point and 10–90 band; **(2)** the
three counterfactuals, one sub-row each within the buyer's row, each with its band, and the
estimated point repeated on every sub-row as an open black ring WITHOUT a bar. The ring is on
the same line as the counterfactual it is read against, so each displacement is a horizontal
distance; its band is already in figure (1). The axis names the quantity; what the point and
the bar are belongs in the caption.

Finally, the draws. **Every regime, the estimated economy included, is redrawn in Python** by
`simulate_granular_regime` from `theta+` — successive replications are successive draws from one
`default_rng` stream, so realisations are independent of one another. (Until the modules were
split, `Both forces` was read off `suppliers.parquet` instead and the two routes had to be read
against each other; now there is one route, and `check_against_julia` is what ties it to Julia.)
Within a replication one Fréchet draw per (variety, cell) is **shared across buyers** — it is
the same physical variety — which is what makes chance common rather than idiosyncratic. So the
realisations are independent and the buyers within one are not, and it is the second that the
per-buyer caveat above is about.


In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
# The local share as a LEVEL (with each force switched off) and then as a level plus a
# DISPERSION. One statistic, read twice.
ls_tabs = {}
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    name = cfg.get("display_name", industry)
    # the ESTIMATED economy, simulated from theta+ -- `reporting_data` caches,
    # so the sections share one solve rather than re-solving each
    data = reporting_data(industry, mu=MU, **RUN_KWARGS)["Both forces"]
    globals().update(data)

    print(f"\n=== [{industry}, mu = {MU}] " + "=" * 46)
    diff = build_diffusion_frame(data)
    summ = amplification_summary(data, radii=AMPLIFICATION_RADII, diffusion=diff)
    prof = local_share_profile(data, diffusion=diff)

    # ---------------------------------------------------------------- the LEVEL
    for d in AMPLIFICATION_RADII:
        c = f"share_within_{int(d)}km"
        print(f"  within {d:g} km: mean {summ[c].mean():.3f}, "
              f"median {summ[c].median():.3f}, min {summ[c].min():.3f}, max {summ[c].max():.3f}")

    plot_local_share(data, radii=AMPLIFICATION_RADII, summary=summ,
                     save_to=f"{out_folder}/local_share_{industry}_mu{MU}.pdf")
    plt.show()
    plot_local_share_profile(data, profile=prof, mark=AMPLIFICATION_RADII,
                             save_to=f"{out_folder}/local_share_profile_{industry}_mu{MU}.pdf")
    plt.show()

    # the map: L_r(d) is a number attached to a place. Guarded on the geometry.
    try:
        plot_local_share_map(data, radius_km=AMPLIFICATION_RADII[-1], summary=summ,
                             vlim=LOCAL_SHARE_MAP_VLIM_WIDE,
                             save_to=f"{out_folder}/local_share_map{int(AMPLIFICATION_RADII[-1])}km_{industry}_mu{MU}.pdf")
        plt.show()
    except (FileNotFoundError, KeyError, ValueError) as e:
        print(f"  local-share map skipped: {type(e).__name__}: {e}")

    # the sign reversal, region by region and as the whole profile. The frames come from
    # the RE-SOLVED economies, so the intermediate expenditure share responds -- see the
    # channel decomposition in test 4.
    frames = simulated_frames(reporting_data(industry, mu=MU, **RUN_KWARGS))
    det = counterfactual_amplification(data, radii=AMPLIFICATION_RADII,
                                       frames=frames, diffusion=diff)
    plot_counterfactual_profile(data, mark=AMPLIFICATION_RADII, diffusion=diff,
                                save_to=f"{out_folder}/counterfactual_profile_{industry}_mu{MU}.pdf")
    plt.show()

    # ----------------------------------------------------------- the DISPERSION
    # 200 km, the section's own radius: the algebra is the same indicator for any radius,
    # and the own-zone share (`radius_km=None`) sits an order of magnitude lower, which
    # flattens the band and leaves the figure mute.
    tag = f"{int(LOCAL_SHARE_RADIUS_KM)}km"
    try:
        # the three forces removed one at a time: area advantage, density gradient, distance
        tab = local_share_dispersion(data, regimes=CF_REGIMES_KAPPA,
                                     radius_km=LOCAL_SHARE_RADIUS_KM)
    except (FileNotFoundError, KeyError, ValueError) as e:
        print(f"  local-share dispersion skipped: {type(e).__name__}: {e}")
        continue
    local_share_report(tab)
    ls_tabs[(name, tag)] = tab

    # Two figures: (1) the estimate alone, black, point + 10-90 band (GRANULARITY at a
    # fixed theta+, never a confidence interval); (2) the three counterfactuals with their
    # bands, the estimate as an open ring without bar on each sub-row. `save_to` is a stem:
    # `_baseline` and `_counterfactuals` are appended.
    plot_local_share_dispersion(
        tab, layout="pair",
        save_to=f"{out_folder}/local_share_dispersion_{tag}_{industry}_mu{MU}.pdf")
    plt.show()

    # the band, and the two numbers that say whether it reads as written: `coverage`
    # (does [q10, q90] hold 80% of the economies, or is it over-covering a lattice)
    # and `p_n_eff` (how fine that lattice is)
    _c = tab.loc["Both forces"]
    display(_c[["local", "q10", "q90", "width", "coverage", "p_n_eff",
                "n_draws", "region"]].round(3))
